# Notebook 10 – Support Vector Machine (SVM)

## Topics
- Support Vector Machine
- Hyperplane
- Margin
- Support Vectors
- Linear SVM
- Non-Linear SVM
- Kernel
- RBF Kernel
- C Parameter
- Gamma
- Feature Scaling
- Intuition behind maximizing the margin

## 1. Support Vector Machine

Support Vector Machine (SVM) is a **supervised machine learning algorithm** mainly used for classification. It finds a decision boundary that separates classes as effectively as possible.

The main idea is to find a boundary with the **maximum possible margin** between the classes.

## 2. Hyperplane

A **hyperplane** is the decision boundary used by SVM to separate classes.

For two features, it is a line. For three features, it is a plane. In higher dimensions, it is called a hyperplane.

For a linear classifier, the general form is:

$w^T x + b = 0$

## 3. Margin

The **margin** is the distance between the decision boundary and the closest data points from each class.

SVM tries to maximize this margin.

### Intuition behind maximizing the margin

Imagine several possible lines can separate two classes. A line very close to one class can be sensitive to small changes in the data. A line with a wider gap from both classes is generally more robust.

Therefore, SVM chooses the boundary that gives the **largest safe separation** between classes. This is called **maximum-margin classification**.

## 4. Support Vectors

**Support vectors** are the training data points closest to the decision boundary.

They are important because they determine the position of the optimal boundary and margin.

Points far away from the boundary usually have much less influence on the final decision boundary.

## 5. Linear SVM

A **Linear SVM** is used when the classes can be separated reasonably well using a straight decision boundary.

In scikit-learn, `SVC(kernel='linear')` creates a linear SVM classifier.

## 6. Non-Linear SVM

A **Non-Linear SVM** is useful when a straight line or plane cannot separate the classes effectively.

SVM can use **kernels** to create flexible decision boundaries without explicitly transforming the data into a very high-dimensional space.

## 7. Kernel

A **kernel** measures similarity between data points and allows SVM to learn non-linear decision boundaries.

Common kernels include:
- Linear
- Polynomial
- RBF
- Sigmoid

The **RBF kernel** is one of the most commonly used kernels for non-linear SVM.

## 8. RBF Kernel

RBF stands for **Radial Basis Function**.

It can create curved and complex decision boundaries by measuring how close data points are to each other.

In scikit-learn:

`SVC(kernel='rbf')`

## 9. C Parameter

`C` controls the trade-off between a wide margin and classification errors.

- **Small C:** allows more training errors and generally gives a wider, smoother margin.
- **Large C:** strongly penalizes training errors and tries to classify training points correctly, which can produce a more complex boundary.

Very large C can increase the risk of overfitting.

## 10. Gamma

`gamma` is mainly important for non-linear kernels such as RBF.

- **Small gamma:** each training point has a broader influence, producing a smoother boundary.
- **Large gamma:** each point has a more local influence, producing a more complex boundary.

Very large gamma can lead to overfitting.

## 11. Feature Scaling

Feature scaling is important for SVM because SVM uses distances and margins.

If one feature has values much larger than another, it can dominate the calculation.

We use `StandardScaler` to standardize the features.

**Important:** fit the scaler only on the training data, then transform both training and test data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

## 12. Load the Dataset

We will use the Iris dataset, a standard classification dataset containing three flower classes.

In [ ]:
iris = load_iris()
X = iris.data
y = iris.target

print('Features shape:', X.shape)
print('Target shape:', y.shape)
print('Classes:', iris.target_names)

## 13. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training samples:', X_train.shape[0])
print('Testing samples:', X_test.shape[0])

## 14. Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Scaling completed.')

## 15. Train a Linear SVM

In [ ]:
linear_svm = SVC(kernel='linear', C=1.0)
linear_svm.fit(X_train_scaled, y_train)

y_pred_linear = linear_svm.predict(X_test_scaled)
linear_accuracy = accuracy_score(y_test, y_pred_linear)

print('Linear SVM Accuracy:', linear_accuracy)

## 16. Train a Non-Linear SVM using RBF Kernel

In [ ]:
rbf_svm = SVC(kernel='rbf', C=1.0, gamma='scale')
rbf_svm.fit(X_train_scaled, y_train)

y_pred_rbf = rbf_svm.predict(X_test_scaled)
rbf_accuracy = accuracy_score(y_test, y_pred_rbf)

print('RBF SVM Accuracy:', rbf_accuracy)

## 17. Classification Report

In [ ]:
print('RBF SVM Classification Report')
print(classification_report(y_test, y_pred_rbf, target_names=iris.target_names))

## 18. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_rbf)
print(cm)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=iris.target_names
).plot()
plt.title('RBF SVM Confusion Matrix')
plt.show()

## 19. Support Vectors

The `n_support_` attribute shows the number of support vectors for each class.

In [ ]:
print('Support vectors per class:', rbf_svm.n_support_)
print('Total support vectors:', len(rbf_svm.support_))

## 20. Effect of C

We compare several C values while keeping the RBF kernel and gamma fixed.

In [ ]:
C_values = [0.01, 0.1, 1, 10, 100]
c_accuracies = []

for c in C_values:
    model = SVC(kernel='rbf', C=c, gamma='scale')
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    c_accuracies.append(accuracy_score(y_test, predictions))

for c, acc in zip(C_values, c_accuracies):
    print(f'C={c}: Accuracy={acc:.3f}')

In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogx(C_values, c_accuracies, marker='o')
plt.xlabel('C')
plt.ylabel('Test Accuracy')
plt.title('Effect of C on RBF SVM')
plt.grid(True)
plt.show()

## 21. Effect of Gamma

We compare different gamma values to see how the complexity of the RBF decision boundary changes.

In [ ]:
gamma_values = [0.01, 0.1, 1, 10, 100]
gamma_accuracies = []

for gamma in gamma_values:
    model = SVC(kernel='rbf', C=1.0, gamma=gamma)
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    gamma_accuracies.append(accuracy_score(y_test, predictions))

for gamma, acc in zip(gamma_values, gamma_accuracies):
    print(f'Gamma={gamma}: Accuracy={acc:.3f}')

In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogx(gamma_values, gamma_accuracies, marker='o')
plt.xlabel('Gamma')
plt.ylabel('Test Accuracy')
plt.title('Effect of Gamma on RBF SVM')
plt.grid(True)
plt.show()

## 22. Linear SVM vs RBF SVM

In [ ]:
print(f'Linear SVM Accuracy: {linear_accuracy:.3f}')
print(f'RBF SVM Accuracy:    {rbf_accuracy:.3f}')

## 23. Advantages of SVM

- Effective for classification problems.
- Works well in high-dimensional feature spaces.
- The maximum-margin idea can provide good generalization.
- Kernel functions allow non-linear classification.
- Only support vectors are critical to the final boundary.

## 24. Limitations of SVM

- Can be slower on very large datasets.
- Feature scaling is usually important.
- Choosing `C`, `gamma`, and the kernel can require tuning.
- Results can be sensitive to noisy or overlapping data.
- The model can be harder to interpret than a decision tree.

# 25. Viva Summary

1. **SVM:** A supervised algorithm that finds a decision boundary for classification.
2. **Hyperplane:** The decision boundary separating classes.
3. **Margin:** The distance between the hyperplane and the nearest data points.
4. **Maximum margin:** SVM tries to maximize the separation between classes to obtain a more robust boundary.
5. **Support vectors:** The closest training points that determine the margin and boundary.
6. **Linear SVM:** Uses a linear decision boundary.
7. **Non-linear SVM:** Uses kernels to learn curved decision boundaries.
8. **Kernel:** A function that measures similarity and enables non-linear classification.
9. **RBF kernel:** A popular non-linear kernel based on the distance between points.
10. **C:** Controls the penalty for classification errors; larger C generally makes the model less tolerant of errors.
11. **Gamma:** Controls how local the influence of individual points is for the RBF kernel.
12. **Small gamma:** Smoother, broader influence.
13. **Large gamma:** More local influence and a more complex boundary.
14. **Feature scaling:** Important because SVM relies on distances and margins.
15. **Overfitting:** Can occur with overly complex boundaries, such as very large C or gamma in some datasets.